# ch2~4

## 활성화 함수 구현 및 속도
- 시그모이드가 좀 느리네. 탄젠트?
- 왜 배치단위로 로스값을 계산할 때, 하나로 합치지? 배치가 아니더라도 그냥 합칠 필요 없이, 각각 아웃풋 층으로만 역전파시키면 안되나?
- 배치단위로 하는 것 역시 그냥 단순히 배치크기로만 나눈다고 이게 될 문제인가?
- 배치 뽑을 때 exhasutive하게 뽑지 않는 문제점?

In [7]:
import numpy as np
from numba import jit, njit

In [8]:
## Textbook clone-coding
def step_function(x: np.ndarray)->np.ndarray:   
    y = x > 0
    return y.astype(int)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def relu(x):
    return np.maximum(0, x)

def softmax(a):
    C = np.max(a)
    exp_a = np.exp(a) - np.max(a)
    sum_exp_a = np.sum(exp_a)
    y = exp_a / sum_exp_a
    return y

In [16]:
## My own functions
## 228 µs ± 22 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
def step_function(x):
    x[x > 0] = 1
    x[x <= 0] = 0
    return x

## 163 µs ± 34.6 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
def sigmoid(x):     
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

## 90.3 µs ± 4.85 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)
def relu(x): 
    x[x <= 0] = 0
    return x

def softmax(x):
    return np.exp(x - np.max(x)) / np.sum(np.exp(x - np.max(x)))



### loss functions ###
def sum_squares_error(y: np.ndarray, t: np.ndarray) -> np.ndarray:
    return 0.5 * np.sum((y - t)**2)

def cross_entropy_error(y: np.ndarray, t: np.ndarray) -> np.ndarray:
    if y.ndim == 1:  ## if it is NOT batch
        y = y.reshape(1, y.size)
        t = t.reshape(1, t.size)
    
    batch_size = y.shape[0]
    
    return -np.sum(t * np.log(y + 1e7)) / batch_size


In [11]:
## 22.8 µs ± 2.43 µs per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
@jit
def step_function(x: np.ndarray)->np.ndarray:
    y = x > 0
    return y.astype(int)

## 193 µs ± 18.3 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
@njit
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

## 12.5 µs ± 12.6 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
@njit
def relu(x):    ## 463
    return np.maximum(0, x)

C:\Users\dieyo\AppData\Local\Temp\ipykernel_11352\3401287908.py:3: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def step_function(x: np.ndarray)->np.ndarray:


In [17]:
arr = np.random.randn(500, 500)
function_tup = (sigmoid, tanh)

for f in (step_function, sigmoid, relu):
    %timeit f(arr)
    print(f(arr), '\n\n')

7.55 ms ± 362 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
[[1. 0. 0. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 [0. 1. 1. ... 1. 1. 0.]
 ...
 [1. 0. 1. ... 1. 1. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 1.]] 


7.19 ms ± 597 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
[[0.73105858 0.5        0.5        ... 0.5        0.5        0.5       ]
 [0.73105858 0.73105858 0.5        ... 0.5        0.5        0.5       ]
 [0.5        0.73105858 0.73105858 ... 0.73105858 0.73105858 0.5       ]
 ...
 [0.73105858 0.5        0.73105858 ... 0.73105858 0.73105858 0.5       ]
 [0.5        0.73105858 0.5        ... 0.5        0.5        0.5       ]
 [0.5        0.73105858 0.5        ... 0.5        0.5        0.73105858]] 


3.42 ms ± 474 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
[[1. 0. 0. ... 0. 0. 0.]
 [1. 1. 0. ... 0. 0. 0.]
 [0. 1. 1. ... 1. 1. 0.]
 ...
 [1. 0. 1. ... 1. 1. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 1.]] 


